## Setup and load data

This notebook answers RQ2: does the shared encoder help or hurt compared to training a separate model per target. I use exactly the same seed, folds, splits and training procedure as the authoritative run in Notebook 6, so the only difference between the two is single-task versus multi-task.

In [1]:
import numpy as np
import pickle
import torch
import torch.nn as nn
import torch.nn.functional as F
import random

SEED=42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic=True
torch.backends.cudnn.benchmark=False

device="cuda" if torch.cuda.is_available() else "cpu"
print("device:",device)
print("gpu:",torch.cuda.get_device_name(0) if device=="cuda" else "none")

DATA="/kaggle/input/datasets/arpitjoshua/mt-trajnet-thesis-data/kaggle_upload"

with open(DATA+"/assembled_trajectories.pkl","rb") as fh:
    data=pickle.load(fh)
X_traj=data["X_traj"]
y_arr=data["y_arr"]
groups=data["groups"]

print("loaded:",len(X_traj),"batches | y",y_arr.shape,"| codes",len(np.unique(groups)))
print("any NaN:",any(np.isnan(a).any() for a in X_traj))

device: cuda
gpu: Tesla T4
loaded: 1005 batches | y (1005, 4) | codes 25
any NaN: False


## Data preparation

Same preparation as the authoritative run: every second step, capped at 6000, with normalisation computed from training data inside each fold.

In [2]:
STRIDE=2
MAXLEN=6000
targets=["dissolution_av","tbl_av_hardness","tbl_rsd_weight","fct_tensile"]

def prep_batch(traj_list,mean,std,maxlen=MAXLEN):
    out=[]
    for a in traj_list:
        a=a[::STRIDE]
        a=(a-mean)/std
        if len(a)<maxlen:
            pad=np.zeros((maxlen-len(a),a.shape[1]),dtype="float32")
            a=np.vstack([a,pad])
        else:
            a=a[:maxlen]
        out.append(a)
    arr=np.array(out,dtype="float32")
    return torch.tensor(arr).transpose(1,2)

covered=sum(1 for a in X_traj if len(a[::STRIDE])<=MAXLEN)
print("stride",STRIDE,"maxlen",MAXLEN)
print("fully covered:",covered,"of",len(X_traj),f"({100*covered/len(X_traj):.0f}%)")

stride 2 maxlen 6000
fully covered: 939 of 1005 (93%)


## Model classes

The encoder and the evidential head are identical to the multi-task model. The only difference is that the single-task model has one evidential head for one target, rather than four heads sharing an encoder.

In [3]:
class TCNBlock(nn.Module):
    def __init__(self,in_ch,out_ch,dilation,kernel=3,dropout=0.1):
        super().__init__()
        pad=(kernel-1)*dilation
        self.conv1=nn.Conv1d(in_ch,out_ch,kernel,padding=pad,dilation=dilation)
        self.conv2=nn.Conv1d(out_ch,out_ch,kernel,padding=pad,dilation=dilation)
        self.relu=nn.ReLU()
        self.drop=nn.Dropout(dropout)
        self.pad=pad
        self.down=nn.Conv1d(in_ch,out_ch,1) if in_ch!=out_ch else None
    def forward(self,x):
        res=x if self.down is None else self.down(x)
        out=self.conv1(x)[:,:,:-self.pad]
        out=self.drop(self.relu(out))
        out=self.conv2(out)[:,:,:-self.pad]
        out=self.drop(self.relu(out))
        return self.relu(out+res)

class TCNEncoder(nn.Module):
    def __init__(self,in_ch=10,hidden=128,dropout=0.1):
        super().__init__()
        self.b1=TCNBlock(in_ch,hidden,1,dropout=dropout)
        self.b2=TCNBlock(hidden,hidden,2,dropout=dropout)
        self.b3=TCNBlock(hidden,hidden,4,dropout=dropout)
        self.b4=TCNBlock(hidden,hidden,8,dropout=dropout)
    def forward(self,x):
        x=self.b1(x);x=self.b2(x);x=self.b3(x);x=self.b4(x)
        return x.mean(dim=2)

class EvidentialHead(nn.Module):
    def __init__(self,in_dim,hidden=64,dropout=0.1):
        super().__init__()
        self.net=nn.Sequential(nn.Linear(in_dim,hidden),nn.ReLU(),nn.Dropout(dropout),nn.Linear(hidden,4))
    def forward(self,x):
        out=self.net(x)
        gamma=out[:,0]
        nu=F.softplus(out[:,1]).clamp(min=1e-2,max=1e3)
        alpha=F.softplus(out[:,2]).clamp(min=1e-2,max=1e3)+1.0
        beta=F.softplus(out[:,3]).clamp(min=1e-2,max=1e3)
        return gamma,nu,alpha,beta

def evidential_loss(y,gamma,nu,alpha,beta,lam=1.0):
    om=2*beta*(1+nu)
    nll=(0.5*torch.log(np.pi/nu)-alpha*torch.log(om)
         +(alpha+0.5)*torch.log(nu*(y-gamma)**2+om)
         +torch.lgamma(alpha)-torch.lgamma(alpha+0.5))
    reg=torch.abs(y-gamma)*(2*nu+alpha)
    return (nll+lam*reg).mean()

class SingleTaskNet(nn.Module):
    def __init__(self,in_ch=10,hidden=128,dropout=0.1):
        super().__init__()
        self.encoder=TCNEncoder(in_ch,hidden,dropout)
        self.head=EvidentialHead(hidden,dropout=dropout)
    def forward(self,x):
        z=self.encoder(x)
        return self.head(z)

m=SingleTaskNet().to(device)
t=torch.randn(3,10,500).to(device)
g,n,a,b=m(t)
var=b/(n*(a-1))
print("prediction shape:",g.shape,"| variance finite:",torch.isfinite(var).all().item())
print("parameters:",sum(p.numel() for p in m.parameters()))

prediction shape: torch.Size([3]) | variance finite: True
parameters: 358852


## Single-task training

I train one model per target, using the same folds, the same seed, and the same three-way split of training products as the multi-task run. The calibration partition is held out even though a single-task model does not need it, because otherwise the single-task models would train on more data than the multi-task model and the comparison would be unfair.

In [4]:
from sklearn.model_selection import GroupKFold
from sklearn.metrics import mean_squared_error
import time,warnings
warnings.filterwarnings("ignore")

def train_single(target_idx,n_splits=3,max_epochs=150,lr=5e-4,batch_size=16,patience=10):
    gkf=GroupKFold(n_splits=n_splits)
    fold_rmse=[]
    for fold,(trv,te) in enumerate(gkf.split(X_traj,y_arr[:,0],groups=groups)):
        g=groups[trv];uniq=np.unique(g)
        rng2=np.random.RandomState(SEED+fold)
        shuf=rng2.permutation(uniq)
        k=max(1,len(shuf)//8)
        val_codes=shuf[:k];cal_codes=shuf[k:2*k]
        tr=trv[~np.isin(g,np.concatenate([val_codes,cal_codes]))]
        va=trv[np.isin(g,val_codes)]

        xmean=np.vstack([X_traj[i][::STRIDE] for i in tr]).mean(axis=0)
        xstd=np.vstack([X_traj[i][::STRIDE] for i in tr]).std(axis=0)+1e-6
        Xtr=prep_batch([X_traj[i] for i in tr],xmean,xstd)
        Xva=prep_batch([X_traj[i] for i in va],xmean,xstd)
        Xte=prep_batch([X_traj[i] for i in te],xmean,xstd)
        ymean=y_arr[tr,target_idx].mean();ystd=y_arr[tr,target_idx].std()+1e-6
        ytr=torch.tensor((y_arr[tr,target_idx]-ymean)/ystd,dtype=torch.float32)
        yva=torch.tensor((y_arr[va,target_idx]-ymean)/ystd,dtype=torch.float32)
        yte=y_arr[te,target_idx]

        torch.manual_seed(SEED+fold)
        model=SingleTaskNet().to(device)
        opt=torch.optim.Adam(model.parameters(),lr=lr)
        best=float("inf");best_state=None;wait=0
        for ep in range(max_epochs):
            model.train()
            perm=torch.randperm(len(Xtr))
            for i in range(0,len(Xtr),batch_size):
                idx=perm[i:i+batch_size]
                xb=Xtr[idx].to(device);yb=ytr[idx].to(device)
                opt.zero_grad()
                gg,nn_,aa,bb=model(xb)
                loss=evidential_loss(yb,gg,nn_,aa,bb)
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(),1.0)
                opt.step()
            model.eval()
            with torch.no_grad():
                vl=0
                for i in range(0,len(Xva),batch_size):
                    xb=Xva[i:i+batch_size].to(device);yb=yva[i:i+batch_size].to(device)
                    gg,nn_,aa,bb=model(xb)
                    vl+=evidential_loss(yb,gg,nn_,aa,bb).item()
            if vl<best-1e-4:
                best=vl;best_state={k3:v.cpu().clone() for k3,v in model.state_dict().items()};wait=0
            else:
                wait+=1
                if wait>=patience:break
        model.load_state_dict(best_state);model.eval()

        preds=[]
        with torch.no_grad():
            for i in range(0,len(Xte),batch_size):
                xb=Xte[i:i+batch_size].to(device)
                gg,_,_,_=model(xb)
                preds.append(gg.cpu().numpy())
        pred=np.concatenate(preds)*ystd+ymean
        fold_rmse.append(np.sqrt(mean_squared_error(yte,pred)))
    return np.array(fold_rmse)

print("ready")

ready


## Train one model per target

Four separate models, one per target, each with its own encoder. Each prints its per-fold RMSE as it finishes.

In [5]:
t0=time.time()
single_folds={}
for j,t in enumerate(targets):
    print(f"training {t} ...")
    r=train_single(target_idx=j)
    single_folds[t]=[float(v) for v in r]
    print(f"  {t}: {r.mean():.3f} (std {r.std():.3f})  folds {[round(v,3) for v in r]}")
print("\ntime:",round(time.time()-t0),"seconds")

training dissolution_av ...
  dissolution_av: 3.926 (std 0.881)  folds [np.float64(3.303), np.float64(3.304), np.float64(5.172)]
training tbl_av_hardness ...
  tbl_av_hardness: 12.733 (std 5.328)  folds [np.float64(17.856), np.float64(5.385), np.float64(14.956)]
training tbl_rsd_weight ...
  tbl_rsd_weight: 0.554 (std 0.107)  folds [np.float64(0.429), np.float64(0.54), np.float64(0.691)]
training fct_tensile ...
  fct_tensile: 0.388 (std 0.143)  folds [np.float64(0.275), np.float64(0.299), np.float64(0.59)]

time: 925 seconds


## RQ2: single-task against multi-task

I compare the per-fold RMSE of the single-task models against the multi-task fold values from the authoritative run in Notebook 6, using the Nadeau-Bengio corrected test. Both sets come from the same folds, the same seed and the same three-way split, so the only difference is the shared encoder. With three folds the test has low power, so p-values are weak evidence in either direction.

In [6]:
from scipy import stats
import json

multi_folds={
"dissolution_av":[4.049,3.428,4.140],
"tbl_av_hardness":[19.713,7.842,11.662],
"tbl_rsd_weight":[0.445,0.528,0.688],
"fct_tensile":[0.376,0.304,0.608],
}

print(f"{'target':<18}{'single':<10}{'multi':<10}{'diff':<10}{'p-value':<10}{'significant'}")
rq2={}
for t in targets:
    s=np.array(single_folds[t]);m=np.array(multi_folds[t])
    d=s-m
    n=len(d);mean_d=d.mean();var_d=d.var(ddof=1)
    cv=var_d*(1/n+0.5)
    tstat=mean_d/np.sqrt(cv) if cv>0 else 0
    p=2*(1-stats.t.cdf(abs(tstat),n-1))
    rq2[t]={"single_folds":[round(v,3) for v in s],"single_mean":round(float(s.mean()),3),
            "multi_folds":m.tolist(),"multi_mean":round(float(m.mean()),3),
            "p":round(float(p),3),"significant":bool(p<0.05)}
    print(f"{t:<18}{s.mean():<10.3f}{m.mean():<10.3f}{mean_d:<10.3f}{p:<10.3f}{'yes' if p<0.05 else 'no'}")

with open("/kaggle/working/rq2_seeded.json","w") as fh:
    json.dump({"comparison":"single-task vs multi-task, both on the seeded pipeline (seed 42), matched GroupKFold(3), Nadeau-Bengio",
               "note":"single-task models hold out the same calibration partition as the multi-task model so both train on identical data; three folds gives low statistical power",
               "results":rq2},fh,indent=2)
print("\nsaved rq2_seeded.json")

target            single    multi     diff      p-value   significant
dissolution_av    3.926     3.872     0.054     0.954     no
tbl_av_hardness   12.733    13.072    -0.340    0.917     no
tbl_rsd_weight    0.554     0.554     -0.000    0.993     no
fct_tensile       0.388     0.429     -0.041    0.474     no

saved rq2_seeded.json


In [7]:
print("RQ2 SUMMARY (seeded pipeline, seed 42)\n")
print(f"{'target':<18}{'single-task':<14}{'multi-task':<14}{'p-value'}")
for t in targets:
    r=rq2[t]
    print(f"{t:<18}{r['single_mean']:<14}{r['multi_mean']:<14}{r['p']}")

single_params=358852
multi_params=384400
print(f"\nparameters: multi-task {multi_params:,} | four single-task models {4*single_params:,}")
print(f"reduction: {4*single_params/multi_params:.2f}x fewer parameters with the shared encoder")
print("\nno significant difference on any target; shared encoder does not harm accuracy")

RQ2 SUMMARY (seeded pipeline, seed 42)

target            single-task   multi-task    p-value
dissolution_av    3.926         3.872         0.954
tbl_av_hardness   12.733        13.072        0.917
tbl_rsd_weight    0.554         0.554         0.993
fct_tensile       0.388         0.429         0.474

parameters: multi-task 384,400 | four single-task models 1,435,408
reduction: 3.73x fewer parameters with the shared encoder

no significant difference on any target; shared encoder does not harm accuracy
